In [ ]:
import sys
import subprocess
import importlib

_required = ["torch", "torchvision", "matplotlib", "numpy<2.0"]
_missing = []
for _mod in _required:
    try:
        importlib.import_module(_mod)
    except ImportError:
        _missing.append(_mod)

if _missing:
    print(f"Installing missing packages: {_missing}")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *_missing])
    except subprocess.CalledProcessError as e:
        print("Package installation failed:", e)
        raise

importlib.invalidate_caches()

# Perform the original imports
import torch.nn as nn

print("All imports succeeded.")

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from model import get_model
import utils

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

In [ ]:
# Load MNIST test dataset
transform = transforms.Compose([
    transforms.ToTensor(),
])

test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Use a subset for faster testing (300 samples)
# For full evaluation, you can use the entire test set (10,000 samples)
test_subset = torch.utils.data.Subset(test_dataset, range(300))
test_loader = torch.utils.data.DataLoader(test_subset, batch_size=300, shuffle=False)

# Get a batch of test data
x_test, y_test = next(iter(test_loader))

# Flatten images from (batch, 1, 28, 28) to (batch, 784)
x_test = x_test.view(x_test.size(0), -1).to(device)
y_test = y_test.to(device)

print(f"Test set size: {x_test.shape[0]}")
print(f"Image shape (flattened): {x_test.shape[1]}")
print(f"Data range: {x_test.min():.3f} to {x_test.max():.3f}")

In [ ]:
# Load pre-trained model
model = get_model(pretrained_path="data/model.pth", device=device)

# Evaluate clean accuracy
clean_acc = utils.accuracy(model, x_test, y_test)
print(f"Clean accuracy (unperturbed): {clean_acc:.4f}")